## MOVIE RECOMMENDER SYSTEM

Recommendation systems are used everywhere including e-commerce websites,streaming platforms like netflix ,spotify etc.

It is of two types:

1) Content-based recommendation system-recommends on the basis of the content,we compare the similarity of the tags associated with them
   
2) Collaborative filtering based-shows videos based on what users like me prefer to consume  

3) Hybrid

##### In this project we will implement collaborative filtering and content based filtering both to show the difference

We will do so by use of a **tmdb** dataset which is basically a database of movies

In [94]:
import numpy as np
import pandas as pd

In [95]:
movies=pd.read_csv("tmdb_5000_movies.csv")
credits=pd.read_csv("tmdb_5000_credits.csv")

In [96]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [97]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


Lets merge the two dataframes and do the data preprocessing on the merged dataset

In [98]:
movies=movies.merge(credits,on='title')

In [99]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [100]:
movies['original_language'].value_counts()

en    4510
fr      70
es      32
zh      27
de      27
hi      19
ja      16
it      14
ko      12
cn      12
ru      11
pt       9
da       7
sv       5
nl       4
fa       4
th       3
he       3
ta       2
cs       2
ro       2
id       2
ar       2
vi       1
sl       1
ps       1
no       1
ky       1
hu       1
pl       1
af       1
nb       1
tr       1
is       1
xx       1
te       1
el       1
Name: original_language, dtype: int64

In [101]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

Lets analyse to see which columns need to be removed and what needs to be kept
When we create a content based recommendation we need to create tags for the movies- o while reviewing a column we need to see if it is playing a part in creating the tags or not

We do not need orginial language as most of the movies in the dataset are in english.

Popularity wont contribute anythng to the content.

We might need release date,popularity but its numerical so its not fitting into our workflow.

**genre,id,keywords,title,overview,cast,crew**

In [102]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [103]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4809 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4809 non-null   int64 
 1   title     4809 non-null   object
 2   overview  4806 non-null   object
 3   genres    4809 non-null   object
 4   keywords  4809 non-null   object
 5   cast      4809 non-null   object
 6   crew      4809 non-null   object
dtypes: int64(1), object(6)
memory usage: 300.6+ KB


##### We will make a new dataframe that will consist of three columns-movie_id,title and tags

##### The tags column can be made by adding genres and overview together and top 3 cast and only the director among the crew

In [104]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [105]:
movies.dropna(inplace=True)

In [106]:
movies.duplicated().sum()

0

In [107]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

We want to convert the above into this format ['action','adventure'.....]

The input is string and that has to converted into list and to do that we will use something like **ast.literal_eval()**

In [108]:
import ast
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [109]:
convert('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [110]:
movies['genres']=movies['genres'].apply(convert)

In [111]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [112]:
movies['keywords']=movies['keywords'].apply(convert)

In [113]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [114]:
movies['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [115]:
#We will only take the top three actors in the movie
import ast
def convert_cast(obj):
    L=[]
    count=0
    for i in ast.literal_eval(obj):
        if count!=3:
            L.append(i['name'])
            count+=1
        else:
            break
    return L

In [116]:
movies['cast']=movies['cast'].apply(convert_cast)

In [117]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [118]:
movies['crew'][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [119]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(i['name'])
            break
    return L
    

In [120]:
movies['crew']=movies['crew'].apply(fetch_director)

In [121]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


#### Creating the tags now

In [122]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [123]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


Now we just have to concantenate the lists and convert them back to string so that we can get the tags

The problem is :
Sam Worthington ->SamWorthington
Because if there are durectors with the same first name , then due to separation of the words the two crews will be treated the same and their will be a confusion.Example,sam mendes and sam worthington

In [124]:
movies['genres']=movies['genres'].apply(lambda x : [i.replace(' ' , '') for i in x])

In [125]:
movies['keywords']=movies['keywords'].apply(lambda x : [i.replace(' ' , '') for i in x])
movies['cast']=movies['cast'].apply(lambda x : [i.replace(' ' , '') for i in x])
movies['crew']=movies['crew'].apply(lambda x : [i.replace(' ' , '') for i in x])

In [126]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]


In [127]:
movies['tag']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [128]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew,tag
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [129]:
new_df=movies[['movie_id','title','tag']]

In [130]:
new_df.head(1)

,movie_id,title,tag
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [131]:
new_df['tag']=new_df['tag'].apply(lambda x: " ".join(x))

/var/folders/f5/ynb377151c7dz7s2d9yt1qs80000gn/T/ipykernel_978/2671889383.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tag']=new_df['tag'].apply(lambda x: " ".join(x))


In [132]:
new_df['tag'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [133]:
new_df['tag']=new_df['tag'].apply(lambda x:x.lower())

/var/folders/f5/ynb377151c7dz7s2d9yt1qs80000gn/T/ipykernel_978/1740795540.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tag']=new_df['tag'].apply(lambda x:x.lower())


In [134]:
new_df.head(1)

,movie_id,title,tag
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


Now, we will do text vectorization to convert the text into vectors and calculate the similarity using cosine similarity between

When we want the model to recommend movies based on some movie it will recommend the closest vectors 

The method that we will use is **Bag of Words**
Here,we will not consider the stopwords during vectorization that can be taken care by the stopwords feature of the count vectorizer class

In [135]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [136]:
ps.stem('loving')

'love'

In [137]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [138]:
new_df['tag']=new_df['tag'].apply(stem)

/var/folders/f5/ynb377151c7dz7s2d9yt1qs80000gn/T/ipykernel_978/599176683.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tag']=new_df['tag'].apply(stem)


In [140]:
new_df['tag'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [142]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [143]:
vectors= cv.fit_transform(new_df['tag']).toarray()

In [144]:
cv.fit_transform(new_df['tag']).toarray().shape

(4806, 5000)

In [145]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

We have a sparse matrix above

In [146]:
len(cv.get_feature_names_out())

5000

In [147]:
cv.vocabulary_

{'century': 740,
 'marin': 2806,
 'dispatch': 1288,
 'moon': 3017,
 'pandora': 3282,
 'uniqu': 4681,
 'mission': 2984,
 'becom': 442,
 'torn': 4554,
 'follow': 1729,
 'order': 3235,
 'protect': 3534,
 'alien': 157,
 'action': 79,
 'adventur': 106,
 'fantasi': 1629,
 'sciencefict': 3930,
 'cultureclash': 1087,
 'futur': 1807,
 'societi': 4153,
 'spacetravel': 4198,
 'futurist': 1809,
 'romanc': 3811,
 'space': 4192,
 'tribe': 4605,
 'alienplanet': 160,
 'soldier': 4160,
 'battl': 425,
 '3d': 47,
 'zoesaldana': 4994,
 'sigourneyweav': 4080,
 'jamescameron': 2325,
 'captain': 680,
 'long': 2695,
 'believ': 455,
 'dead': 1152,
 'ha': 1960,
 'come': 917,
 'life': 2647,
 'head': 2018,
 'edg': 1408,
 'earth': 1396,
 'turner': 4628,
 'elizabeth': 1436,
 'noth': 3172,
 'quit': 3572,
 'ocean': 3198,
 'drugabus': 1358,
 'exoticisland': 1571,
 'loveofone': 2729,
 'slif': 4132,
 'traitor': 4584,
 'shipwreck': 4056,
 'ship': 4055,
 'allianc': 167,
 'afterlif': 119,
 'fighter': 1680,
 'pirat': 3391,


In [148]:
len(cv.vocabulary_)

5000

We will calculate the distance between two vectors and greater the distance lesser is the similarity. And instead of eucleidian distance we will calculate the cosine distance that basically calculates the angle differene=ce instead of the tip to tip vector distance like in euclidean distance
In higher dimensions ,eucleidian distance is not a good measure-curse of dimensionality

We will use a function is scikit-learn called **cosine similarity**

In [150]:
from sklearn.metrics.pairwise import cosine_similarity

In [151]:
cosine_similarity(vectors)


array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]])

This returns a matrix where the distance between every pair of vectors is calculated.The diagonal elements are all zero because the cosine of the distance ebweetn the same vectors is zero

In [153]:
cosine_similarity(vectors).shape

(4806, 4806)

In [154]:
similarity=cosine_similarity(vectors)

In [155]:
similarity[0]

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ])

In [157]:
similarity[0].shape

(4806,)

We will create a function that takes in th name of the movie and returns the most similar 5 movies based on cosine similarity
For that we will have to fetch the index of the move and apply the method accordingly

In [164]:
new_df[new_df['title']=='Batman Begins']

,movie_id,title,tag
119,272,Batman Begins,"driven by tragedy, billionair bruce wayn dedic..."


In [165]:
new_df[new_df['title']=='Batman Begins'].index

Int64Index([119], dtype='int64')

In [166]:
new_df[new_df['title']=='Batman Begins'].index[0]

119

In [169]:
new_df.iloc[119].title

'Batman Begins'

In [167]:
list(enumerate(similarity[0]))

[(0, 1.0000000000000002),
 (1, 0.08346223261119858),
 (2, 0.08603090020146065),
 (3, 0.0734718358370645),
 (4, 0.1892994097121204),
 (5, 0.10838874619051501),
 (6, 0.04024218182927669),
 (7, 0.14673479641335554),
 (8, 0.05923488777590923),
 (9, 0.0967301666813349),
 (10, 0.10259783520851541),
 (11, 0.09464970485606021),
 (12, 0.09037128496931669),
 (13, 0.04499212706658476),
 (14, 0.12824729401064427),
 (15, 0.06282808624375433),
 (16, 0.07894736842105264),
 (17, 0.13977653617040256),
 (18, 0.09493290614465533),
 (19, 0.0830812984794528),
 (20, 0.058038100008800934),
 (21, 0.10968169942141635),
 (22, 0.0662266178532522),
 (23, 0.08740748201220976),
 (24, 0.0533380747062665),
 (25, 0.05101627678885769),
 (26, 0.15389675281277312),
 (27, 0.18693292157876878),
 (28, 0.116543309349613),
 (29, 0.065033247714309),
 (30, 0.06684847767323797),
 (31, 0.15907119074394446),
 (32, 0.08520286456846099),
 (33, 0.09733285267845754),
 (34, 0.0),
 (35, 0.09933992677987831),
 (36, 0.17316974359835272),


In [172]:
def recommend(movie):
    #fetch the index of the movie
    movie_index=new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6] 
    #we have to sort in suvh a way that even after sorting the indexes remain in place
    #and sorting has to dine on the basis of the first number so we mention the key
    #take 5 movies from 1 , because we dont want the same movie in the recommendation

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [173]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [174]:
import pickle

In [177]:
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))

In [178]:
pickle.dump(similarity,open('similarity.pkl','wb'))